# Analyze per Die Matrix

This notebook demonstrates how to find the **reference behavior** for each die matrix, compare individual pieces against it, and identify which process segments show the most variability.

All analysis is performed on the gold parquet dataset (clean, validated pieces with partial times).

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

gold_path = Path("../data/gold/pieces.parquet")
if not gold_path.exists():
    raise FileNotFoundError(f"Gold parquet not found: {gold_path.resolve()}")

df = pd.read_parquet(gold_path)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

print("Gold dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

cumulative_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
]

partial_cols = [
    "partial_furnace_to_2nd_strike_s",
    "partial_2nd_to_3rd_strike_s",
    "partial_3rd_to_4th_strike_s",
    "partial_4th_strike_to_auxiliary_press_s",
    "partial_auxiliary_press_to_bath_s",
]

display(df.head(10))

Gold dataset shape: (140404, 18)
Columns: ['timestamp', 'piece_id', 'die_matrix', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_bath_s', 'lifetime_general_s', 'partial_furnace_to_2nd_strike_s', 'partial_2nd_to_3rd_strike_s', 'partial_3rd_to_4th_strike_s', 'partial_4th_strike_to_auxiliary_press_s', 'partial_auxiliary_press_to_bath_s', 'oee_cycle_time_s', 'seconds_since_prev_piece', 'is_production_gap', 'production_run_id']


,timestamp,piece_id,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s,oee_cycle_time_s,seconds_since_prev_piece,is_production_gap,production_run_id
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001,17.900000,6.700001,13.400000,16.599998,1.600002,NaN,NaN,False,1
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002,17.900000,6.700001,13.300001,16.899998,1.600002,NaN,12.708,False,1
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002,18.200001,6.599998,13.500000,17.000000,1.600002,NaN,14.609,False,1
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998,18.400000,6.700001,13.300001,17.099998,1.599998,NaN,12.719,False,1
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001,18.200001,6.599998,13.400002,17.299999,1.700001,NaN,14.000,False,1
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,23.400000,36.799999,53.599998,55.200001,55.200001,16.700001,6.699999,13.400000,16.799999,1.600002,13.4690,13.309,False,1
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,24.900000,38.200001,54.700001,56.400002,56.400002,18.299999,6.600000,13.300001,16.500000,1.700001,13.4496,12.611,False,1
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,24.500000,37.700001,54.700001,56.299999,56.299999,17.900000,6.600000,13.200001,17.000000,1.599998,13.2704,13.713,False,1
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,23.299999,36.599998,53.299999,54.900002,54.900002,16.700001,6.599998,13.299999,16.700001,1.600002,NaN,67.332,False,1
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,23.600000,36.799999,54.099998,55.799999,55.799999,16.700001,6.900000,13.199999,17.299999,1.700001,NaN,67.043,False,1


## 1. Reference profile per die matrix

The **reference (optimal) behavior** for each die matrix is defined by its median cumulative times. The median is more robust than the mean — it is not pulled by residual edge cases.

This table shows how long a typical piece takes to reach each stage, per matrix.

In [3]:
reference_cumulative = (
    df.groupby("die_matrix")[cumulative_cols]
    .median()
    .round(2)
    .reset_index()
)

display(reference_cumulative)


,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s
0,4974,17.3,23.9,37.1,54.2,56.0
1,5052,18.3,25.3,39.3,56.7,58.3
2,5090,17.7,24.6,38.5,56.5,58.1
3,5091,17.9,24.6,38.2,55.2,56.8


## 2. Reference partial times per die matrix

The partial times (time spent **between** consecutive stages) are more diagnostically useful than cumulative times — they isolate each segment of the process.

In [4]:
reference_partials = (
    df.groupby("die_matrix")[partial_cols]
    .median()
    .round(2)
    .reset_index()
)

display(reference_partials)

,die_matrix,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s
0,4974,17.3,6.5,13.1,17.0,1.8
1,5052,18.3,6.9,13.7,17.3,1.6
2,5090,17.7,6.8,13.8,17.7,1.6
3,5091,17.9,6.6,13.5,17.0,1.6


## 3. Variability per segment per die matrix

Which segments are most variable? High standard deviation relative to the median (coefficient of variation) indicates segments where the process is less stable — these are the primary candidates for delay investigation.

**CV = std / median × 100%** — higher CV means more variability relative to the typical value.

In [5]:
variability_summary = []

for matrix, group in df.groupby("die_matrix"):
    for col in partial_cols:
        median_val = group[col].median()
        std_val = group[col].std()
        cv_val = (std_val / median_val * 100) if pd.notna(median_val) and median_val != 0 else np.nan

        variability_summary.append({
            "die_matrix": matrix,
            "segment": col,
            "median_s": round(median_val, 2) if pd.notna(median_val) else np.nan,
            "std_s": round(std_val, 2) if pd.notna(std_val) else np.nan,
            "cv_pct": round(cv_val, 2) if pd.notna(cv_val) else np.nan,
        })

variability_summary = pd.DataFrame(variability_summary)

display(variability_summary)

most_variable_segments = (
    variability_summary.sort_values(["die_matrix", "cv_pct"], ascending=[True, False])
    .groupby("die_matrix")
    .head(1)
    .reset_index(drop=True)
)

display(most_variable_segments)


,die_matrix,segment,median_s,std_s,cv_pct
0,4974,partial_furnace_to_2nd_strike_s,17.3,1.58,9.13
1,4974,partial_2nd_to_3rd_strike_s,6.5,0.27,4.08
2,4974,partial_3rd_to_4th_strike_s,13.1,0.31,2.36
3,4974,partial_4th_strike_to_auxiliary_press_s,17.0,0.88,5.21
4,4974,partial_auxiliary_press_to_bath_s,1.8,0.05,2.70
5,5052,partial_furnace_to_2nd_strike_s,18.3,1.91,10.43
6,5052,partial_2nd_to_3rd_strike_s,6.9,0.50,7.29
7,5052,partial_3rd_to_4th_strike_s,13.7,0.85,6.23
8,5052,partial_4th_strike_to_auxiliary_press_s,17.3,1.15,6.63
9,5052,partial_auxiliary_press_to_bath_s,1.6,0.05,3.07


,die_matrix,segment,median_s,std_s,cv_pct
0,4974,partial_furnace_to_2nd_strike_s,17.3,1.58,9.13
1,5052,partial_furnace_to_2nd_strike_s,18.3,1.91,10.43
2,5090,partial_furnace_to_2nd_strike_s,17.7,2.40,13.55
3,5091,partial_furnace_to_2nd_strike_s,17.9,2.45,13.67


## 4. Deviation from reference per piece

For each piece, compute the deviation from its die matrix reference at each stage. Positive deviation = slower than reference. This allows identifying both slow individual pieces and systematic drifts.

In [6]:
# Merge matrix-level reference partial medians back to each piece
ref_partials = (
    df.groupby("die_matrix")[partial_cols]
    .median()
    .reset_index()
)

df = df.merge(ref_partials, on="die_matrix", suffixes=("", "_ref"))

# Compute per-piece deviation from matrix reference
for col in partial_cols:
    df[f"{col}_dev"] = df[col] - df[f"{col}_ref"]

dev_cols = [f"{col}_dev" for col in partial_cols]

display(
    df[
        ["timestamp", "piece_id", "die_matrix"] +
        partial_cols +
        [f"{c}_ref" for c in partial_cols] +
        dev_cols
    ].head(10)
)


,timestamp,piece_id,die_matrix,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s,partial_furnace_to_2nd_strike_s_ref,partial_2nd_to_3rd_strike_s_ref,partial_3rd_to_4th_strike_s_ref,partial_4th_strike_to_auxiliary_press_s_ref,partial_auxiliary_press_to_bath_s_ref,partial_furnace_to_2nd_strike_s_dev,partial_2nd_to_3rd_strike_s_dev,partial_3rd_to_4th_strike_s_dev,partial_4th_strike_to_auxiliary_press_s_dev,partial_auxiliary_press_to_bath_s_dev
0,2025-11-06 15:25:16.426000+00:00,251106001721,5052,17.900000,6.700001,13.400000,16.599998,1.600002,18.299999,6.9,13.700001,17.299999,1.600002,-0.400000,-0.199999,-0.300001,-0.700001,0.000000
1,2025-11-06 15:25:29.134000+00:00,251106001722,5052,17.900000,6.700001,13.300001,16.899998,1.600002,18.299999,6.9,13.700001,17.299999,1.600002,-0.400000,-0.199999,-0.400000,-0.400002,0.000000
2,2025-11-06 15:25:43.743000+00:00,251106001723,5052,18.200001,6.599998,13.500000,17.000000,1.600002,18.299999,6.9,13.700001,17.299999,1.600002,-0.099998,-0.300001,-0.200001,-0.299999,0.000000
3,2025-11-06 15:25:56.462000+00:00,251106001724,5052,18.400000,6.700001,13.300001,17.099998,1.599998,18.299999,6.9,13.700001,17.299999,1.600002,0.100000,-0.199999,-0.400000,-0.200001,-0.000004
4,2025-11-06 15:26:10.462000+00:00,251106001725,5052,18.200001,6.599998,13.400002,17.299999,1.700001,18.299999,6.9,13.700001,17.299999,1.600002,-0.099998,-0.300001,-0.299999,0.000000,0.099998
5,2025-11-06 15:26:23.771000+00:00,251106001726,5052,16.700001,6.699999,13.400000,16.799999,1.600002,18.299999,6.9,13.700001,17.299999,1.600002,-1.599998,-0.200001,-0.300001,-0.500000,0.000000
6,2025-11-06 15:26:36.382000+00:00,251106001727,5052,18.299999,6.600000,13.300001,16.500000,1.700001,18.299999,6.9,13.700001,17.299999,1.600002,0.000000,-0.299999,-0.400000,-0.799999,0.099998
7,2025-11-06 15:26:50.095000+00:00,251106001728,5052,17.900000,6.600000,13.200001,17.000000,1.599998,18.299999,6.9,13.700001,17.299999,1.600002,-0.400000,-0.299999,-0.500000,-0.299999,-0.000004
8,2025-11-06 15:27:57.427000+00:00,251106001733,5052,16.700001,6.599998,13.299999,16.700001,1.600002,18.299999,6.9,13.700001,17.299999,1.600002,-1.599998,-0.300001,-0.400002,-0.599998,0.000000
9,2025-11-06 15:29:04.470000+00:00,251106001738,5052,16.700001,6.900000,13.199999,17.299999,1.700001,18.299999,6.9,13.700001,17.299999,1.600002,-1.599998,0.000000,-0.500002,0.000000,0.099998


## 5. Identify slow pieces and their penalized segment

A piece is considered **slow** if its total bath time exceeds the 90th percentile for its die matrix. For each slow piece, identify which segment contributed the most delay.

In [7]:
# Bath-time 90th percentile per die matrix
bath_p90 = (
    df.groupby("die_matrix")["lifetime_bath_s"]
    .quantile(0.90)
    .reset_index()
    .rename(columns={"lifetime_bath_s": "bath_p90_s"})
)

df = df.merge(bath_p90, on="die_matrix", how="left")

# Slow piece = bath time above matrix-specific 90th percentile
df["is_slow_piece"] = df["lifetime_bath_s"] > df["bath_p90_s"]

# Penalized segment = segment with the largest positive deviation
df["penalized_segment"] = df[dev_cols].idxmax(axis=1).str.replace("_dev", "", regex=False)

print("Total slow pieces:", int(df["is_slow_piece"].sum()))
print("Slow piece share (%):", round(df["is_slow_piece"].mean() * 100, 2))

display(
    df.loc[df["is_slow_piece"], [
        "timestamp",
        "piece_id",
        "die_matrix",
        "lifetime_bath_s",
        "bath_p90_s",
        "penalized_segment",
    ] + dev_cols].head(10)
)


Total slow pieces: 13758
Slow piece share (%): 9.8


,timestamp,piece_id,die_matrix,lifetime_bath_s,bath_p90_s,penalized_segment,partial_furnace_to_2nd_strike_s_dev,partial_2nd_to_3rd_strike_s_dev,partial_3rd_to_4th_strike_s_dev,partial_4th_strike_to_auxiliary_press_s_dev,partial_auxiliary_press_to_bath_s_dev
114,2025-11-06 15:56:48.815000+00:00,251106001861,5052,67.099998,61.799999,partial_4th_strike_to_auxiliary_press_s,-1.000000,3.100000,-0.400002,7.500004,0.099995
225,2025-11-06 16:59:53.995000+00:00,251106002034,5052,63.700001,61.799999,partial_4th_strike_to_auxiliary_press_s,-1.000000,0.400002,-0.200003,6.700001,0.000000
235,2025-11-06 17:02:37.141000+00:00,251106002046,5052,64.699997,61.799999,partial_4th_strike_to_auxiliary_press_s,0.300001,-0.200001,-0.299999,7.099998,-0.000004
938,2025-11-06 20:06:41.332000+00:00,251106002864,5052,67.099998,61.799999,partial_4th_strike_to_auxiliary_press_s,-0.400000,-0.299999,-0.200001,10.100002,0.099995
956,2025-11-06 20:21:07.831000+00:00,251106002929,5052,62.000000,61.799999,partial_2nd_to_3rd_strike_s,-0.199999,4.400000,-0.100000,0.000000,0.099998
1068,2025-11-06 23:05:02.464000+00:00,251106003159,5052,62.599998,61.799999,partial_furnace_to_2nd_strike_s,5.000000,-0.299999,-0.100000,0.200001,-0.000004
1071,2025-11-06 23:05:43.489000+00:00,251106003162,5052,62.000000,61.799999,partial_furnace_to_2nd_strike_s,4.800001,-0.299999,-0.200001,-0.099998,-0.000004
1073,2025-11-06 23:06:10.819000+00:00,251106003164,5052,62.299999,61.799999,partial_furnace_to_2nd_strike_s,5.100000,-0.199999,-0.100000,-0.299999,-0.000004
1075,2025-11-06 23:06:37.537000+00:00,251106003166,5052,62.599998,61.799999,partial_furnace_to_2nd_strike_s,5.300001,-0.200001,-0.099998,-0.200001,-0.000004
1076,2025-11-06 23:06:51.043000+00:00,251106003167,5052,62.200001,61.799999,partial_furnace_to_2nd_strike_s,4.900002,-0.200001,-0.199999,-0.100002,0.000000


## 6. Slow pieces per die matrix

How slow pieces distribute across die matrices, and which segments are most often penalized per matrix.

In [8]:
slow_piece_summary = (
    df.groupby("die_matrix")
    .agg(
        n_pieces=("piece_id", "size"),
        n_slow_pieces=("is_slow_piece", "sum"),
    )
    .reset_index()
)

slow_piece_summary["slow_piece_pct"] = (
    slow_piece_summary["n_slow_pieces"] / slow_piece_summary["n_pieces"] * 100
).round(2)

display(slow_piece_summary)

penalized_segment_summary = (
    df.loc[df["is_slow_piece"]]
    .groupby(["die_matrix", "penalized_segment"])
    .size()
    .reset_index(name="count")
    .sort_values(["die_matrix", "count"], ascending=[True, False])
)

display(penalized_segment_summary)


,die_matrix,n_pieces,n_slow_pieces,slow_piece_pct
0,4974,15531,1527,9.83
1,5052,20887,2081,9.96
2,5090,81677,7946,9.73
3,5091,22309,2204,9.88


,die_matrix,penalized_segment,count
3,4974,partial_furnace_to_2nd_strike_s,1328
2,4974,partial_4th_strike_to_auxiliary_press_s,177
1,4974,partial_3rd_to_4th_strike_s,18
0,4974,partial_2nd_to_3rd_strike_s,4
7,5052,partial_furnace_to_2nd_strike_s,1708
6,5052,partial_4th_strike_to_auxiliary_press_s,191
5,5052,partial_3rd_to_4th_strike_s,152
4,5052,partial_2nd_to_3rd_strike_s,30
12,5090,partial_furnace_to_2nd_strike_s,5798
9,5090,partial_3rd_to_4th_strike_s,1236


## 7. Time evolution — detecting drift

Does the process get slower over time for a given matrix? Plot the daily median bath time per matrix to detect progressive deterioration.

In [9]:
df["production_date"] = pd.to_datetime(df["timestamp"]).dt.date

daily_bath_by_matrix = (
    df.groupby(["die_matrix", "production_date"])["lifetime_bath_s"]
    .median()
    .reset_index()
    .rename(columns={"lifetime_bath_s": "daily_median_bath_s"})
    .sort_values(["die_matrix", "production_date"])
)

display(daily_bath_by_matrix.head(20))

drift_summary = []

for matrix, group in daily_bath_by_matrix.groupby("die_matrix"):
    group = group.sort_values("production_date").reset_index(drop=True)

    start_period = group.head(5)["daily_median_bath_s"].mean()
    end_period = group.tail(5)["daily_median_bath_s"].mean()

    drift_summary.append({
        "die_matrix": matrix,
        "start_period_median_bath_s": round(start_period, 2),
        "end_period_median_bath_s": round(end_period, 2),
        "delta_end_minus_start_s": round(end_period - start_period, 2),
    })

drift_summary = pd.DataFrame(drift_summary).sort_values("die_matrix").reset_index(drop=True)

display(drift_summary)

,die_matrix,production_date,daily_median_bath_s
0,4974,2025-11-13,56.299999
1,4974,2025-11-14,57.599998
2,4974,2025-11-16,57.500000
3,4974,2025-11-17,55.599998
4,4974,2025-11-18,56.599998
5,4974,2025-11-19,56.099998
6,4974,2025-11-20,55.900002
7,4974,2025-11-21,56.400002
8,4974,2025-11-23,58.500000
9,4974,2025-11-24,55.700001


,die_matrix,start_period_median_bath_s,end_period_median_bath_s,delta_end_minus_start_s
0,4974,56.72,56.46,-0.26
1,5052,57.02,59.85,2.83
2,5090,56.70,61.84,5.14
3,5091,56.39,59.23,2.84


## 8. Segment variability ranking

Across all matrices, which process segment is the most unstable? This is where maintenance and process engineering should focus attention.

In [10]:
segment_variability_ranking = (
    variability_summary.groupby("segment")
    .agg(
        mean_cv_pct=("cv_pct", "mean"),
        median_cv_pct=("cv_pct", "median"),
        max_cv_pct=("cv_pct", "max"),
    )
    .round(2)
    .reset_index()
    .sort_values("mean_cv_pct", ascending=False)
    .reset_index(drop=True)
)

display(segment_variability_ranking)


,segment,mean_cv_pct,median_cv_pct,max_cv_pct
0,partial_furnace_to_2nd_strike_s,11.70,11.99,13.67
1,partial_2nd_to_3rd_strike_s,8.41,9.08,11.40
2,partial_4th_strike_to_auxiliary_press_s,6.77,6.37,9.13
3,partial_3rd_to_4th_strike_s,6.44,7.16,9.07
4,partial_auxiliary_press_to_bath_s,4.56,4.24,7.09


## 9. Summary

Key findings from the per-matrix analysis.

## Summary

### Reference behavior

Each die matrix exhibits a distinct timing profile, confirming that all analysis must be performed per matrix. Median cumulative and partial times define the expected "normal" behavior of a piece traveling through the line.

---

### Segment variability

The most variable segment across matrices is:

- **Furnace → 2nd strike** (mean CV ≈ 11.7%)

This segment shows significantly higher variability than all others, indicating that:

- Robot pick and transfer from furnace to main press is the least stable part of the process
- Variability likely comes from positioning, retries, interlocks, or queueing effects

Downstream segments (especially auxiliary press → bath) are much more stable, with CV below ~5%.

---

### Slow pieces

- Total slow pieces: **13,758**
- Share of total: **~9.8%**

This aligns closely with the 90th percentile definition, confirming correct implementation.

Across all die matrices, the proportion of slow pieces is consistent (~9.7–9.9%), indicating that:

- No single matrix is disproportionately problematic in terms of overall slow-piece frequency

---

### Root cause of delays (penalized segments)

For all matrices, the dominant penalized segment is:

- **Furnace → 2nd strike**

This segment accounts for the majority of delays:
- Matrix 5090: 5,798 cases
- Matrix 5052: 1,708 cases
- Matrix 5091: 1,614 cases
- Matrix 4974: 1,328 cases

This clearly identifies the **entry stage of the main press** as the primary bottleneck.

Secondary contributors vary by matrix but include:
- 3rd → 4th strike (internal press transfer)
- 4th strike → auxiliary press (exit + deburring transfer)

---

### Drift over time

Significant drift is observed for some matrices:

- **Matrix 5090:** +5.14s increase in median bath time
- **Matrix 5052 / 5091:** ~+2.8s increase
- **Matrix 4974:** stable (−0.26s)

This suggests:

- Progressive degradation (likely tooling wear or process inefficiency buildup) for matrices 5052, 5090, and 5091
- Matrix 5090 shows the strongest evidence of deterioration

---

### Key insights

- Process performance is **matrix-specific** and must be analyzed accordingly  
- The **furnace → main press transfer** is the most unstable and delay-prone segment  
- Slow pieces are driven by **localized segment delays**, not uniform slowdowns  
- Some matrices exhibit **clear temporal drift**, indicating potential maintenance or calibration needs  

---

### Operational implication

The highest-impact improvement opportunity lies in:

- Stabilizing the furnace → main press transfer (robot handling, positioning, interlocks)
- Monitoring drift in matrices showing degradation (especially 5090)

These actions would directly reduce slow-piece frequency and improve overall line consistency.
